# Ausgrid experiment 2: paired conditional diffusion

This is a **follow-on to `Ausgrid_Colab.ipynb`**. Keep that notebook and its completed VAE runs. Put this notebook in the same `notebooks/` directory of the existing project. It reads `data/prepared/prepared.npz`, uses the **same split and conditions**, and writes only new diffusion checkpoints, samples, and reports under `outputs/`. Three seeds compare `diffusion_post` with your existing `vae_post`. An unmasked `diffusion` arm uses the *same sampled profiles* and tests the effect of the daylight mask.

Use a Colab GPU runtime. The first cell finds the project from the notebook folder or mounted Drive and mounts Drive when needed. No new CSV processing or source-code installation is needed. The model generates **independent paired days**, not continuous synthetic households. The household membership AUC is a bounded empirical probe, not a privacy guarantee.

Motivation: [Fu et al. (2024)](https://arxiv.org/abs/2404.00525) studied conditional diffusion for energy meter synthesis. This Ausgrid comparison is a separate experiment with less metadata and a different target.


In [ ]:
from pathlib import Path
import os, sys, json, math, random, time, hashlib, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

workdir = Path.cwd().resolve()
candidates = [workdir, workdir.parent, workdir/'ausgrid-synthetic',
              Path('/content/drive/MyDrive/ausgrid-synthetic')]
PROJECT_ROOT = next((p for p in candidates if (p/'data/prepared/prepared.npz').is_file()
                     and (p/'src/ausgrid_synth/evaluate.py').is_file()), None)
if PROJECT_ROOT is None:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/ausgrid-synthetic')
assert (PROJECT_ROOT / 'src/ausgrid_synth/evaluate.py').is_file(), 'Use the project from the original Colab notebook.'
assert (PROJECT_ROOT / 'data/prepared/prepared.npz').is_file(), 'Run the original notebook through Checkpoint 1 first.'
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from ausgrid_synth.data import load_prepared
from ausgrid_synth import evaluate as existing_evaluation

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE.type == 'cuda', 'Use a Colab GPU runtime for diffusion training and sampling.'
DATA = load_prepared(PROJECT_ROOT / 'data/prepared/prepared.npz')
assert DATA['x'].ndim == 3 and DATA['x'].shape[1:] == (2,48)
assert sorted(np.unique(DATA['split']).tolist()) == [0,1,2]
print('GPU:', torch.cuda.get_device_name(0))
print('Paired days:', len(DATA['x']), '| Household split:',
      {name: len(np.unique(DATA['customer'][DATA['split']==code])) for code,name in enumerate(('train','val','test'))})

# Fingerprint the prepared input to prevent resuming against changed data.
hash_obj = hashlib.sha256()
with (PROJECT_ROOT/'data/prepared/prepared.npz').open('rb') as f:
    for block in iter(lambda: f.read(4*1024*1024), b''):
        hash_obj.update(block)
DATA_SHA256 = hash_obj.hexdigest()


## Fixed experiment settings

Both channels are learned **jointly** as 96 values. Apply `log1p(interval kWh / training channel scale)` to moderate peaks. Train a noise-prediction diffusion model on training households, select its epoch with validation households, and sample by deterministic DDIM. A cosine 200-step noise schedule with 32 sampling steps is fixed for this first comparison. Diffusion and VAE need not have identical optimizers, but they use the same calendar/capacity inputs, household split, sample contexts, and evaluation code. The only difference between `diffusion` and `diffusion_post` is the mask applied *after* sampling.

**Primary comparisons:** load/solar daily W1, upper-tail peak errors, morning-to-afternoon real-test MAE. Report nighttime GG and household attack AUC alongside them. Three seeds capture training variability; the held-out households are still a single fixed split.


In [ ]:
CFG = dict(timesteps=200, ddim_steps=32, hidden=256, depth=3, batch_size=512,
           learning_rate=1e-3, patience=5, epochs=25, transform='log1p_energy_by_training_scale',
           model_version=1)
SEEDS = (1, 2, 3)
ARM = 'diffusion'
MASKED_ARM = 'diffusion_post'
MODEL_DIR = PROJECT_ROOT/'outputs/checkpoints'
SAMPLE_DIR = PROJECT_ROOT/'outputs/samples'
REPORT_DIR = PROJECT_ROOT/'outputs/reports'
for path in (MODEL_DIR, SAMPLE_DIR, REPORT_DIR): path.mkdir(parents=True, exist_ok=True)
print('Settings:', CFG)


## Model and cosine schedule

The denoiser takes a noisy 96-value paired day, the same four standardized conditions as the VAE, and a sinusoidal time embedding. It predicts Gaussian noise at a randomly selected diffusion step. Its output is unconstrained during training; nonnegativity and nighttime zero are applied to generated profiles after sampling. The capacity rating is a condition, **not** a strict energy cap.


In [ ]:
def time_embedding(t, dimension=32):
    half = dimension // 2
    frequencies = torch.exp(-math.log(10000) * torch.arange(half, device=t.device, dtype=torch.float32) / (half - 1))
    angle = t.float()[:, None] * frequencies[None, :]
    return torch.cat((angle.sin(), angle.cos()), dim=1)

class PairedDenoiser(nn.Module):
    def __init__(self, hidden=256, depth=3, timesteps=200):
        super().__init__()
        self.timesteps = timesteps
        grid = torch.linspace(0, timesteps, timesteps + 1, dtype=torch.float64)
        cumulative = torch.cos(((grid / timesteps + .008) / 1.008) * math.pi / 2).square()
        cumulative = cumulative / cumulative[0]
        betas = (1 - cumulative[1:] / cumulative[:-1]).clamp(0, .999).float()
        self.register_buffer('alpha_bar', torch.cumprod(1 - betas, dim=0))
        self.time_net = nn.Sequential(nn.Linear(32, hidden), nn.SiLU(), nn.Linear(hidden, hidden))
        self.input_net = nn.Linear(96 + 4 + hidden, hidden)
        self.blocks = nn.ModuleList([nn.Sequential(nn.Linear(hidden, hidden), nn.SiLU(), nn.Linear(hidden, hidden))
                                     for _ in range(depth)])
        self.norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(depth)])
        self.output_net = nn.Linear(hidden, 96)

    def forward(self, x, t, cond):
        emb = self.time_net(time_embedding(t))
        h = F.silu(self.input_net(torch.cat((x, cond, emb), dim=1)))
        for block, norm in zip(self.blocks, self.norms):
            h = norm(h + block(h))
        return self.output_net(F.silu(h))

    def loss(self, x0, cond, generator=None):
        t = torch.randint(0, self.timesteps, (len(x0),), device=x0.device, generator=generator)
        eps = torch.randn(x0.shape, device=x0.device, generator=generator)
        abar = self.alpha_bar[t, None]
        xt = abar.sqrt() * x0 + (1 - abar).sqrt() * eps
        return F.mse_loss(self(xt, t, cond), eps)

assert PairedDenoiser(timesteps=CFG['timesteps']).alpha_bar[-1].item() < 1e-4
print('Final signal fraction:', PairedDenoiser(timesteps=CFG['timesteps']).alpha_bar[-1].item())


## Checkpoint 1 — short pilot, then full training

This training function writes `diffusion_seedN_latest.pt` **at the end of each epoch** and keeps the best-validation `diffusion_seedN.pt`. The latest file contains optimizer and random states; rerunning continues at the next epoch. `status.json` records a finished run. The two-epoch pilot is continued by the 25-epoch run without restarting. If you change `CFG` or prepared data, use a new arm name or remove only the **diffusion** checkpoints and outputs after recording the old run.


In [ ]:
def transformed_dataset(code):
    selected = DATA['split'] == code
    x = np.log1p(DATA['x'][selected] / DATA['scale'][None,:,None]).reshape(-1,96).astype('float32')
    c = DATA['cond'][selected].astype('float32')
    return TensorDataset(torch.from_numpy(x), torch.from_numpy(c))

def safe_save_torch(obj, path):
    tmp = path.with_suffix('.tmp')
    torch.save(obj, tmp)
    tmp.replace(path)

def fit_diffusion(seed, epochs):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    model = PairedDenoiser(CFG['hidden'], CFG['depth'], CFG['timesteps']).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['learning_rate'])
    loader_rng = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(transformed_dataset(0), batch_size=CFG['batch_size'], shuffle=True,
                              generator=loader_rng, pin_memory=True, num_workers=0)
    val_loader = DataLoader(transformed_dataset(1), batch_size=CFG['batch_size'], shuffle=False,
                            pin_memory=True, num_workers=0)
    best_path = MODEL_DIR/f'diffusion_seed{seed}.pt'
    latest_path = MODEL_DIR/f'diffusion_seed{seed}_latest.pt'
    status_path = MODEL_DIR/f'diffusion_seed{seed}_status.json'
    start, best, stale = 0, float('inf'), 0
    if latest_path.exists():
        state = torch.load(latest_path, map_location='cpu', weights_only=False)  # own trusted checkpoint
        if state['config'] != CFG or state['data_sha256'] != DATA_SHA256 or state['seed'] != seed:
            raise ValueError('Existing diffusion checkpoint uses different settings or prepared data.')
        model.load_state_dict(state['model']); optimizer.load_state_dict(state['optimizer'])
        for slot in optimizer.state.values():
            for name, value in slot.items():
                if isinstance(value, torch.Tensor) and name != 'step': slot[name] = value.to(DEVICE)
        start, best, stale = state['epoch'] + 1, state['best'], state['stale']
        random.setstate(state['python_rng']); np.random.set_state(state['numpy_rng'])
        torch.set_rng_state(state['torch_rng']); torch.cuda.set_rng_state_all(state['cuda_rng'])
        loader_rng.set_state(state['loader_rng'])
        print('Resuming seed', seed, 'at epoch', start + 1)
    if start >= epochs:
        print('Seed', seed, 'has already completed', start, 'epochs.'); return
    for epoch in range(start, epochs):
        model.train(); training_sum=0.0; training_n=0
        for x, cond in train_loader:
            x, cond = x.to(DEVICE, non_blocking=True), cond.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            loss = model.loss(x, cond)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            training_sum += loss.item()*len(x); training_n += len(x)
        model.eval(); validation_sum=0.0; validation_n=0
        val_rng = torch.Generator(device=DEVICE).manual_seed(930001)  # fixed validation noise every epoch
        with torch.no_grad():
            for x, cond in val_loader:
                x, cond = x.to(DEVICE, non_blocking=True), cond.to(DEVICE, non_blocking=True)
                loss = model.loss(x, cond, generator=val_rng)
                validation_sum += loss.item()*len(x); validation_n += len(x)
        validation_loss = validation_sum/validation_n
        if validation_loss < best - 1e-5:
            best, stale = validation_loss, 0
            safe_save_torch({'model': model.state_dict(), 'config': CFG, 'seed': seed,
                             'data_sha256': DATA_SHA256, 'best_validation': best}, best_path)
        else: stale += 1
        latest = {'model': model.state_dict(), 'optimizer': optimizer.state_dict(), 'config': CFG,
                  'data_sha256': DATA_SHA256, 'seed': seed, 'epoch': epoch, 'best': best, 'stale': stale,
                  'python_rng': random.getstate(), 'numpy_rng': np.random.get_state(),
                  'torch_rng': torch.get_rng_state(), 'cuda_rng': torch.cuda.get_rng_state_all(),
                  'loader_rng': loader_rng.get_state()}
        safe_save_torch(latest, latest_path)
        status = {'seed': seed, 'last_epoch': epoch+1, 'target_epochs': epochs, 'early_stopped': stale >= CFG['patience'],
                  'best_validation': best, 'data_sha256': DATA_SHA256, 'config': CFG}
        status_path.with_suffix('.tmp').write_text(json.dumps(status,indent=2))
        status_path.with_suffix('.tmp').replace(status_path)
        print(f"seed={seed} epoch={epoch+1} train={training_sum/training_n:.5f} val={validation_loss:.5f} best={best:.5f}", flush=True)
        if stale >= CFG['patience']: break

pilot_start = time.time()
for seed in (1,):
    status_path = MODEL_DIR/f'diffusion_seed{seed}_status.json'
    if not status_path.exists(): fit_diffusion(seed, epochs=2)
print('Pilot elapsed minutes:', round((time.time()-pilot_start)/60,1))


### Full runs: three seeds

After the pilot, inspect its runtime. Adjusting the architecture or schedule now requires resetting only the diffusion checkpoints and noting the change. The default run continues seed 1, then trains seeds 2 and 3. Re-executing this cell skips finished seeds.


In [ ]:
for seed in SEEDS:
    status_path = MODEL_DIR/f'diffusion_seed{seed}_status.json'
    status = json.loads(status_path.read_text()) if status_path.exists() else {}
    complete = (status.get('target_epochs') == CFG['epochs'] and
                (status.get('early_stopped') or status.get('last_epoch',0) >= CFG['epochs']))
    if complete and status.get('config') == CFG and status.get('data_sha256') == DATA_SHA256:
        print('Already finished seed', seed)
    else:
        fit_diffusion(seed, CFG['epochs'])


## Checkpoint 2 — matched sampling

For fair comparison, recreate the exact three context index sets used by the original CLI: 30,000 training days, all held-out days, and 8,000 shared contexts for the attack. Confirm indices against a previous `vae_post` sample when present. DDIM uses 32 deterministic reverse steps after the initial seeded Gaussian noise. Each diffusion sample is saved twice, before and after masking, with an identical internal index. The postprocessed files are for **local evaluation only** because their `idx` links to household data in the prepared file.


In [ ]:
def context_indices(context):
    rng = np.random.default_rng(2026)
    if context == 'train':
        available = np.flatnonzero(DATA['split'] == 0)
        return rng.choice(available, size=min(30000,len(available)), replace=False)
    if context == 'test': return np.flatnonzero(DATA['split'] == 2)
    if context == 'privacy':
        available = np.flatnonzero(np.isin(DATA['split'],[0,2]))
        return rng.choice(available, size=min(8000,len(available)), replace=False)
    raise ValueError(context)

for context in ('train','test','privacy'):
    reference_path = SAMPLE_DIR/f'vae_post_seed1_{context}.npz'
    if reference_path.exists():
        with np.load(reference_path, allow_pickle=False) as reference:
            assert np.array_equal(context_indices(context),reference['idx']), f'Context differs from VAE: {context}'
    print(context, len(context_indices(context)), 'contexts')

@torch.inference_mode()
def generate_diffusion(seed, indices, batch_size=2048):
    best_path = MODEL_DIR/f'diffusion_seed{seed}.pt'
    checkpoint = torch.load(best_path, map_location='cpu', weights_only=False)  # own trusted checkpoint
    if checkpoint['data_sha256'] != DATA_SHA256 or checkpoint['config'] != CFG:
        raise ValueError('Diffusion model does not match this dataset/configuration.')
    model = PairedDenoiser(CFG['hidden'], CFG['depth'], CFG['timesteps']).to(DEVICE)
    model.load_state_dict(checkpoint['model']); model.eval()
    steps = np.rint(np.linspace(CFG['timesteps']-1, 0, CFG['ddim_steps'])).astype(int)
    assert len(np.unique(steps)) == len(steps) and steps[-1] == 0
    generator = torch.Generator(device=DEVICE).manual_seed(2026+seed)
    batches=[]
    for batch_no, start in enumerate(range(0,len(indices),batch_size)):
        chunk = indices[start:start+batch_size]
        cond = torch.from_numpy(DATA['cond'][chunk].astype('float32')).to(DEVICE)
        x = torch.randn((len(chunk),96),device=DEVICE,generator=generator)
        for j,step in enumerate(steps):
            current = torch.full((len(chunk),),int(step),device=DEVICE,dtype=torch.long)
            eps = model(x,current,cond)
            a = model.alpha_bar[int(step)]
            clean = (x - (1-a).sqrt()*eps) / a.sqrt()
            clean = clean.clamp(-8,8)  # numerical safeguard; disclose extreme output below
            if j+1 < len(steps):
                next_a = model.alpha_bar[int(steps[j+1])]
                x = next_a.sqrt()*clean + (1-next_a).sqrt()*eps  # DDIM, eta=0
            else: x = clean
        batches.append(x.reshape(-1,2,48).cpu().numpy())
        if batch_no % 20 == 0: print('generated',min(start+batch_size,len(indices)),'/',len(indices),flush=True)
    model.cpu(); del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    latent = np.concatenate(batches)
    if not np.isfinite(latent).all(): raise ValueError('Nonfinite diffusion output; inspect training.')
    # Energy must be nonnegative. This clipping is independent of the night mask.
    energy = np.expm1(np.clip(latent,0,8)) * DATA['scale'][None,:,None]
    return energy.astype('float32')

def write_sample(path, x, idx):
    path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix('.tmp')
    with tmp.open('wb') as f: np.savez_compressed(f,x=x,idx=idx)
    tmp.replace(path)

for seed in SEEDS:
    for context in ('train','test','privacy'):
        indices=context_indices(context)
        raw_path=SAMPLE_DIR/f'{ARM}_seed{seed}_{context}.npz'
        masked_path=SAMPLE_DIR/f'{MASKED_ARM}_seed{seed}_{context}.npz'
        if raw_path.exists() and masked_path.exists():
            print('Samples already present:',seed,context)
            continue
        if raw_path.exists():
            with np.load(raw_path,allow_pickle=False) as z:
                assert np.array_equal(z['idx'],indices)
                x=z['x'].copy()
        else:
            x=generate_diffusion(seed,indices)
            print('Max interval energy (GC+CL, GG):',np.max(x,axis=(0,2)))
            write_sample(raw_path,x,indices)
        x[:,1,:][DATA['night'][indices]] = 0.0
        write_sample(masked_path,x,indices)
        print('Saved both arms:',seed,context)


## Checkpoint 3 — locked evaluation and upper tails

Run the **same existing evaluator** that produced the VAE table. It reports fidelity, physical violations, synthetic-to-real morning/afternoon utility, and the household attack. The next table also surfaces peak W1 and 95th-percentile peak errors already stored in those reports. Lower W1/MAE is better; the AUC column is a measurement of one attack, not a privacy guarantee.


In [ ]:
for seed in SEEDS:
    for arm in (ARM,MASKED_ARM):
        path=REPORT_DIR/f'{arm}_seed{seed}.json'
        if not path.exists(): existing_evaluation.evaluate(DATA,SAMPLE_DIR,REPORT_DIR,arm,seed)

rows=[]
for arm in ('vae_post',ARM,MASKED_ARM):
    for seed in SEEDS:
        report_path=REPORT_DIR/f'{arm}_seed{seed}.json'
        if not report_path.exists():
            print('Missing previous VAE report:',report_path.name)
            continue
        r=json.loads(report_path.read_text())
        f,u,p=r['fidelity_and_physics'],r['utility'],r['disclosure_probe']
        rows.append({'arm':arm,'seed':seed,
                     'night_GG_kWh':f['night_solar_kwh'],
                     'load_daily_W1':f['total_consumption_daily_kwh_wasserstein'],
                     'solar_daily_W1':f['gross_generation_daily_kwh_wasserstein'],
                     'load_peak_W1':f['total_consumption_peak_interval_kwh_wasserstein'],
                     'solar_peak_W1':f['gross_generation_peak_interval_kwh_wasserstein'],
                     'load_peak_q95_gap':f['total_consumption_peak_interval_kwh_synthetic_q95']-f['total_consumption_peak_interval_kwh_real_q95'],
                     'solar_peak_q95_gap':f['gross_generation_peak_interval_kwh_synthetic_q95']-f['gross_generation_peak_interval_kwh_real_q95'],
                     'GC_GG_total_corr_gap':(f['gccl_gg_daily_total_correlation_synthetic']-
                                            f['gccl_gg_daily_total_correlation_real']),
                     'synthetic_to_real_MAE':u['synthetic_train_test_real_mae'],
                     'real_to_real_MAE':u['real_train_test_real_mae'],
                     'attack_AUC':p['household_membership_auc']})
results=pd.DataFrame(rows)
display(results.round(4))
if len(results):
    display(results.groupby('arm').mean(numeric_only=True).drop(columns='seed').round(4))


## Review profile shapes and outliers

Inspect the median daily shapes, the 95th-percentile peaks, and any extreme single-interval outputs. If diffusion improves daily W1 but worsens upper tails or utility, report that tradeoff instead of selecting it by one metric. The choice between `diffusion` and `diffusion_post` isolates the daylight mask; the choice between `vae_post` and `diffusion_post` compares generators under the same final constraint.


In [ ]:
with np.load(SAMPLE_DIR/'diffusion_post_seed1_test.npz',allow_pickle=False) as z:
    synthetic,indices=z['x'],z['idx']
real=DATA['x'][indices]
for channel,label in enumerate(('GC+CL','GG')):
    real_peak=real[:,channel].max(axis=1)
    fake_peak=synthetic[:,channel].max(axis=1)
    print(label,'interval peak percentiles real / generated:',
          np.quantile(real_peak,[.50,.95,.99]).round(3),
          np.quantile(fake_peak,[.50,.95,.99]).round(3),
          '| maximum generated:',round(float(fake_peak.max()),3))
fig,axes=plt.subplots(1,2,figsize=(12,3))
clock=np.arange(48)/2+0.25
for channel,label in enumerate(('GC+CL','GG')):
    axes[channel].plot(clock,np.median(real[:,channel],axis=0),label='held-out real')
    axes[channel].plot(clock,np.median(synthetic[:,channel],axis=0),label='diffusion + mask',ls='--')
    axes[channel].set(xlabel='Local hour',ylabel='kWh / half-hour',title=label)
    axes[channel].legend()
plt.tight_layout();plt.show()


### Decision record

Compare the **three paired seeds** and the peak/utility tradeoffs before declaring an improved method. The held-out test set is the same for all runs, so repeat seeds are **not** independent household samples. Report the selected mask arm, daily and peak metrics, real-trained utility reference, attack limitations, and any departures from the fixed config. Keep all internal `idx` files and the prepared household data out of public releases.
